# Predicting Smartphone Addiction
### Kaggle Competition - End-to-End Classification Pipeline
**Model Architecture:** Dual-Seed Bagged 10-Fold Grandmaster Ensemble (60 Models: 20x `LightGBM` + 20x `XGBoost` + 20x `HistGradientBoostingClassifier`)  
**Feature Engineering:** 75 Advanced Behavioral, Ratio, Temporal, Non-linear, and Clustered Features  
**Validation Metric:** Out-Of-Fold ROC-AUC, Accuracy, F1-Score, Precision, Recall


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
import xgboost as xgb
import warnings
import time

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Load Data
We load the training and test datasets. The training set consists of 691,369 samples and the test set has 296,302 samples.


In [2]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

print(f"Training Data Shape: {train_df.shape}")
print(f"Test Data Shape:     {test_df.shape}")
train_df.head()


## 2. Advanced Feature Engineering (75 Features)
We construct domain-specific indicators of phone usage patterns:
- **Numerical Encodings & Missingness:** Mappings for gender, academic impact, stress, and total missing value count per row.
- **Component Breakdowns:** Accounted vs unaccounted screen time, entertainment hours (social + gaming), non-study screen hours, and unproductive-to-productive ratios.
- **Awake Time & Sleep Ratios:** Daily awake hours, free awake hours, screen-to-awake ratio, work-to-sleep ratio, sleep deficit, and night screen risk.
- **Micro-Interaction Rates:** App opens per awake hour, notifications per awake hour, compulsive check rate, and minutes per app open.
- **Weekend Dynamics:** Weekend vs daily difference, weekend-to-weekday ratio, and total weekly screen time.
- **Extreme Habit Flags:** Extreme screen time (>12h), high screen + low sleep, severe sleep debt (<4.5h), hyper-connectivity, and binge gamer/social flags.
- **Non-Linear Interactions & Composite Risk:** Addiction risk index v2, multiplicative interaction terms, and quadratic transforms.
- **Age Group Baselines & Frequency Encodings:** Age-group normalized screen deviations and value frequency distributions.
- **User Archetype Clustering:** MiniBatchKMeans (8 clusters) on standardized multi-dimensional behavioral metrics.


In [3]:
def extract_features(train, test):
    df_all = pd.concat([train.assign(is_train=1), test.assign(is_train=0, addicted_label=-1)], ignore_index=True)
    eps = 1e-5
    
    # 1. Numerical encodings
    gender_map = {'Female': 0, 'Male': 1, 'Other': 2}
    df_all['gender_num'] = df_all['gender'].map(gender_map)
    df_all['academic_impact_num'] = df_all['academic_work_impact'].astype(str).str.strip().str.lower().map({'no': 0, 'yes': 1})
    df_all['stress_num'] = df_all['stress_level'].astype(str).str.strip().str.lower().map({'low': 0, 'medium': 1, 'high': 2})
    
    # Stress + Academic interaction category
    df_all['stress_academic_combo'] = df_all['stress_num'].fillna(-1).astype(int).astype(str) + "_" + df_all['academic_impact_num'].fillna(-1).astype(int).astype(str)
    combo_map = {val: i for i, val in enumerate(df_all['stress_academic_combo'].unique())}
    df_all['stress_academic_code'] = df_all['stress_academic_combo'].map(combo_map)
    df_all.drop(columns=['stress_academic_combo'], inplace=True)
    
    # Missing value count
    raw_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
                'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
                'gender', 'stress_level', 'academic_work_impact']
    df_all['num_missing'] = df_all[raw_cols].isnull().sum(axis=1)
    
    # 2. Time component breakdowns
    df_all['accounted_screen_time'] = df_all['social_media_hours'] + df_all['gaming_hours'] + df_all['work_study_hours']
    df_all['unaccounted_screen_time'] = df_all['daily_screen_time_hours'] - df_all['accounted_screen_time']
    df_all['unaccounted_ratio'] = df_all['unaccounted_screen_time'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['entertainment_hours'] = df_all['social_media_hours'] + df_all['gaming_hours']
    df_all['non_study_screen_hours'] = df_all['daily_screen_time_hours'] - df_all['work_study_hours']
    
    # 3. Usage ratios
    df_all['social_media_ratio'] = df_all['social_media_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['gaming_ratio'] = df_all['gaming_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['work_study_ratio'] = df_all['work_study_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['entertainment_ratio'] = df_all['entertainment_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['unproductive_to_productive'] = df_all['entertainment_hours'] / (df_all['work_study_hours'] + eps)
    df_all['social_to_gaming_ratio'] = df_all['social_media_hours'] / (df_all['gaming_hours'] + eps)
    
    # 4. Awake & Sleep dynamics
    df_all['awake_hours'] = 24.0 - df_all['sleep_hours']
    df_all['free_awake_hours'] = (df_all['awake_hours'] - df_all['work_study_hours']).clip(lower=0.1)
    df_all['screen_to_free_awake_ratio'] = df_all['entertainment_hours'] / (df_all['free_awake_hours'] + eps)
    df_all['screen_time_to_awake_ratio'] = df_all['daily_screen_time_hours'] / (df_all['awake_hours'] + eps)
    df_all['non_study_to_sleep_ratio'] = df_all['non_study_screen_hours'] / (df_all['sleep_hours'] + eps)
    df_all['sleep_to_awake_ratio'] = df_all['sleep_hours'] / (df_all['awake_hours'] + eps)
    df_all['screen_to_sleep_ratio'] = df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)
    df_all['work_to_sleep_ratio'] = df_all['work_study_hours'] / (df_all['sleep_hours'] + eps)
    df_all['social_to_awake_ratio'] = df_all['social_media_hours'] / (df_all['awake_hours'] + eps)
    df_all['gaming_to_awake_ratio'] = df_all['gaming_hours'] / (df_all['awake_hours'] + eps)
    df_all['sleep_deficit'] = (8.0 - df_all['sleep_hours']).clip(lower=0)
    df_all['night_screen_risk'] = (df_all['daily_screen_time_hours'] > df_all['awake_hours'] * 0.5).astype(int)
    
    # 5. Micro-interactions & Compulsion metrics
    df_all['notifications_per_awake_hour'] = df_all['notifications_per_day'] / (df_all['awake_hours'] + eps)
    df_all['app_opens_per_awake_hour'] = df_all['app_opens_per_day'] / (df_all['awake_hours'] + eps)
    df_all['notifications_per_app_open'] = df_all['notifications_per_day'] / (df_all['app_opens_per_day'] + eps)
    df_all['minutes_per_app_open'] = (df_all['daily_screen_time_hours'] * 60.0) / (df_all['app_opens_per_day'] + eps)
    df_all['notif_per_screen_minute'] = df_all['notifications_per_day'] / (df_all['daily_screen_time_hours'] * 60.0 + eps)
    df_all['compulsive_check_rate'] = df_all['app_opens_per_day'] / (df_all['daily_screen_time_hours'] * 60.0 + eps)
    
    # 6. Weekend dynamics
    df_all['weekend_vs_daily_diff'] = df_all['weekend_screen_time'] - df_all['daily_screen_time_hours']
    df_all['weekend_vs_daily_ratio'] = df_all['weekend_screen_time'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['weekend_to_weekday_ratio'] = (df_all['weekend_screen_time'] / 2.0) / (df_all['daily_screen_time_hours'] + eps)
    df_all['weekend_to_weekday_diff'] = (df_all['weekend_screen_time'] / 2.0) - df_all['daily_screen_time_hours']
    df_all['total_weekly_screen_time'] = (df_all['daily_screen_time_hours'] * 5.0) + (df_all['weekend_screen_time'] * 2.0)
    
    # 7. Non-linear transforms
    df_all['log_notifications'] = np.log1p(df_all['notifications_per_day'].clip(lower=0))
    df_all['log_app_opens'] = np.log1p(df_all['app_opens_per_day'].clip(lower=0))
    df_all['log_screen_time'] = np.log1p(df_all['daily_screen_time_hours'].clip(lower=0))
    df_all['screen_sleep_sq'] = (df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)) ** 2
    
    # 8. Behavioral thresholds & Risk flags
    df_all['high_screen_low_sleep'] = ((df_all['daily_screen_time_hours'] >= 8) & (df_all['sleep_hours'] <= 5)).astype(int)
    df_all['high_notif_high_open'] = ((df_all['notifications_per_day'] >= 100) & (df_all['app_opens_per_day'] >= 80)).astype(int)
    df_all['severe_impact_stress'] = ((df_all['academic_impact_num'] == 1) & (df_all['stress_num'] == 2)).astype(int)
    df_all['high_screen_flag'] = (df_all['daily_screen_time_hours'] > 9).astype(int)
    df_all['extreme_screen_flag'] = (df_all['daily_screen_time_hours'] > 12).astype(int)
    df_all['severe_sleep_debt'] = (df_all['sleep_hours'] < 4.5).astype(int)
    df_all['hyper_connected'] = (df_all['notifications_per_day'] > 150).astype(int)
    df_all['binge_gamer'] = (df_all['gaming_hours'] > 5).astype(int)
    df_all['binge_social'] = (df_all['social_media_hours'] > 6).astype(int)
    df_all['unproductive_night_owl'] = ((df_all['sleep_hours'] <= 5) & (df_all['entertainment_hours'] >= 7)).astype(int)
    
    # 9. Composite risk scores & cross products
    df_all['screen_stress_inter'] = df_all['daily_screen_time_hours'] * (df_all['stress_num'] + 1)
    df_all['screen_sleep_comp'] = df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)
    df_all['social_sleep_comp'] = df_all['social_media_hours'] / (df_all['sleep_hours'] + eps)
    df_all['notif_app_open_inter'] = df_all['notifications_per_day'] * df_all['app_opens_per_day']
    df_all['addiction_index_v2'] = (2.0 * df_all['non_study_screen_hours'] + 1.5 * df_all['entertainment_hours'] + 0.05 * df_all['notifications_per_day']) / (df_all['sleep_hours'] + 1.0)
    df_all['addiction_risk_score'] = (
        (df_all['daily_screen_time_hours'] > 7).astype(int) +
        (df_all['sleep_hours'] < 6).astype(int) +
        (df_all['social_media_hours'] > 4).astype(int) +
        (df_all['academic_impact_num'] == 1).astype(int) +
        (df_all['stress_num'] == 2).astype(int)
    )
    
    # 10. Age group aggregated features
    df_all['age_group'] = (df_all['age'] // 5) * 5
    age_screen_mean = df_all.groupby('age_group')['daily_screen_time_hours'].transform('mean')
    age_screen_std = df_all.groupby('age_group')['daily_screen_time_hours'].transform('std')
    df_all['screen_time_vs_age_mean'] = df_all['daily_screen_time_hours'] - age_screen_mean
    df_all['screen_time_age_zscore'] = df_all['screen_time_vs_age_mean'] / (age_screen_std + eps)
    
    age_notif_mean = df_all.groupby('age_group')['notifications_per_day'].transform('mean')
    df_all['notif_vs_age_mean'] = df_all['notifications_per_day'] - age_notif_mean
    
    # 11. Frequency encodings
    for col in ['age', 'notifications_per_day', 'app_opens_per_day']:
        freq = df_all[col].value_counts(normalize=True)
        df_all[f'{col}_freq'] = df_all[col].map(freq)
        
    # 12. Clustering archetypes
    cluster_features = ['daily_screen_time_hours', 'social_media_hours', 'sleep_hours', 'notifications_per_day']
    cluster_imputed = df_all[cluster_features].fillna(df_all[cluster_features].median())
    scaler = StandardScaler()
    scaled_feats = scaler.fit_transform(cluster_imputed)
    
    kmeans = MiniBatchKMeans(n_clusters=8, random_state=42, batch_size=2048)
    df_all['user_cluster'] = kmeans.fit_predict(scaled_feats)
    
    df_all = df_all.drop(columns=['gender', 'academic_work_impact', 'stress_level', 'age_group'])
    
    train_res = df_all[df_all['is_train'] == 1].drop(columns=['is_train'])
    test_res = df_all[df_all['is_train'] == 0].drop(columns=['is_train', 'addicted_label'])
    
    return train_res, test_res

train_f, test_f = extract_features(train_df, test_df)
feature_cols = [c for c in train_f.columns if c not in ['id', 'addicted_label']]
print(f"Total features engineered: {len(feature_cols)}")
X = train_f[feature_cols]
y = train_f['addicted_label']
X_test = test_f[feature_cols]


## 3. Dual-Seed Bagged 10-Fold Stratified Triple Ensemble (60 Total Models)
To maximize generalization and minimize variance across tree splits, we employ **Dual-Seed Bagging** (Seed 42 + Seed 2026) across 10 Stratified Folds.
Each fold fits three distinct gradient boosted architectures:
1. **LightGBM Classifier**: `num_leaves=190`, `max_depth=12`, `learning_rate=0.028`, `feature_fraction=0.65`, `bagging_fraction=0.85`
2. **XGBoost Classifier**: `max_depth=9`, `learning_rate=0.030`, `colsample_bytree=0.65`, `subsample=0.85`
3. **HistGradientBoostingClassifier**: `max_iter=380`, `learning_rate=0.045`, `max_leaf_nodes=150`, `l2_regularization=0.8`

**Ensemble Weights:** 45% LightGBM + 40% XGBoost + 15% HistGBM, averaged across both random seeds.


In [4]:
seeds = [42, 2026]
w_lgb = 0.45
w_xgb = 0.40
w_hgb = 0.15

all_test_preds = np.zeros(len(test_df))
all_oof_preds = np.zeros(len(train_df))

for s_idx, seed in enumerate(seeds):
    print(f"\n{'='*30} SEED {seed} ({s_idx+1}/{len(seeds)}) {'='*30}")
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(train_df))
    seed_test = np.zeros(len(test_df))
    
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': 0.028,
        'num_leaves': 190,
        'max_depth': 12,
        'min_child_samples': 25,
        'feature_fraction': 0.65,
        'bagging_fraction': 0.85,
        'bagging_freq': 1,
        'n_estimators': 1000,
        'random_state': seed,
        'n_jobs': -1,
        'verbose': -1
    }
    
    xgb_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'learning_rate': 0.030,
        'max_depth': 9,
        'colsample_bytree': 0.65,
        'subsample': 0.85,
        'n_estimators': 750,
        'tree_method': 'hist',
        'random_state': seed,
        'n_jobs': -1
    }
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        f_start = time.time()
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # 1. LightGBM
        m_lgb = lgb.LGBMClassifier(**lgb_params)
        m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
        val_lgb = m_lgb.predict_proba(X_val)[:, 1]
        t_lgb = m_lgb.predict_proba(X_test)[:, 1]
        
        # 2. XGBoost
        m_xgb = xgb.XGBClassifier(**xgb_params)
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        val_xgb = m_xgb.predict_proba(X_val)[:, 1]
        t_xgb = m_xgb.predict_proba(X_test)[:, 1]
        
        # 3. HistGBM
        m_hgb = HistGradientBoostingClassifier(max_iter=380, learning_rate=0.045, max_leaf_nodes=150, l2_regularization=0.8, random_state=seed+fold)
        m_hgb.fit(X_tr, y_tr)
        val_hgb = m_hgb.predict_proba(X_val)[:, 1]
        t_hgb = m_hgb.predict_proba(X_test)[:, 1]
        
        fold_blend = (val_lgb * w_lgb) + (val_xgb * w_xgb) + (val_hgb * w_hgb)
        seed_oof[val_idx] = fold_blend
        
        fold_test_blend = (t_lgb * w_lgb) + (t_xgb * w_xgb) + (t_hgb * w_hgb)
        seed_test += fold_test_blend / 10.0
        
        auc_f = roc_auc_score(y_val, fold_blend)
        f_elapsed = time.time() - f_start
        print(f"Seed {seed} Fold {fold+1:02d}/10 ({f_elapsed:.1f}s) -> Blend AUC: {auc_f:.5f}")
        
    seed_auc = roc_auc_score(y, seed_oof)
    print(f"--> SEED {seed} FULL 10-FOLD OOF ROC-AUC: {seed_auc:.5f}")
    
    all_oof_preds += seed_oof / len(seeds)
    all_test_preds += seed_test / len(seeds)


============================== SEED 42 (1/2) ==============================
Seed 42 Fold 01/10 (154.4s) -> Blend AUC: 0.96367
Seed 42 Fold 02/10 (160.6s) -> Blend AUC: 0.96416
Seed 42 Fold 03/10 (158.5s) -> Blend AUC: 0.96460
Seed 42 Fold 04/10 (162.4s) -> Blend AUC: 0.96463
Seed 42 Fold 05/10 (162.4s) -> Blend AUC: 0.96417
Seed 42 Fold 06/10 (161.8s) -> Blend AUC: 0.96492
Seed 42 Fold 07/10 (157.2s) -> Blend AUC: 0.96521
Seed 42 Fold 08/10 (159.8s) -> Blend AUC: 0.96562
Seed 42 Fold 09/10 (230.7s) -> Blend AUC: 0.96521
Seed 42 Fold 10/10 (350.6s) -> Blend AUC: 0.96345
--> SEED 42 FULL 10-FOLD OOF ROC-AUC: 0.96456

============================== SEED 2026 (2/2) ==============================
Seed 2026 Fold 01/10 (154.7s) -> Blend AUC: 0.96423
Seed 2026 Fold 02/10 (156.2s) -> Blend AUC: 0.96348
Seed 2026 Fold 03/10 (151.6s) -> Blend AUC: 0.96559
Seed 2026 Fold 04/10 (153.5s) -> Blend AUC: 0.96472
Seed 2026 Fold 05/10 (150.2s) -> Blend AUC: 0.96565
Seed 2026 Fold 06/10 (157.2s) -> Blend 

## 4. Evaluation and Overall Benchmark Metrics
We compute out-of-fold cross-validation performance across the entire 691,369 dataset for the bagged ensemble.


In [5]:
final_auc = roc_auc_score(y, all_oof_preds)
oof_binary = (all_oof_preds >= 0.5).astype(int)
final_acc = accuracy_score(y, oof_binary)
final_f1 = f1_score(y, oof_binary)
final_prec = precision_score(y, oof_binary)
final_rec = recall_score(y, oof_binary)

print("="*78)
print("     DUAL-SEED GRANDMASTER 10-FOLD ENSEMBLE RESULTS     ")
print("="*78)
print(f"DUAL-SEED 60-MODEL OOF ROC-AUC:      {final_auc:.5f}")
print(f"Accuracy:                            {final_acc*100:.2f}%")
print(f"F1-Score:                            {final_f1:.5f}")
print(f"Precision:                           {final_prec:.5f}")
print(f"Recall:                              {final_rec:.5f}")
print("="*78)


     DUAL-SEED GRANDMASTER 10-FOLD ENSEMBLE RESULTS (57.0 mins)     
DUAL-SEED 60-MODEL OOF ROC-AUC:      0.96466
Accuracy:                            90.37%
F1-Score:                            0.93255
Precision:                           0.92727
Recall:                              0.93789


## 5. Generate Submission
We ensemble the 60 out-of-fold test predictions and write the output to `submission.csv`.


In [6]:
sub_df = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': all_test_preds
})

sub_df.to_csv('submission.csv', index=False)
print(f"Submission saved to submission.csv with {len(sub_df):,} rows.")
print(sub_df.head(10))


Submission saved to submission.csv with 296,302 rows.
       id  addicted_label
0  691369        0.999036
1  691370        0.951360
2  691371        0.943453
3  691372        0.992671
4  691373        0.996986
5  691374        0.999754
6  691375        0.998412
7  691376        0.988350
8  691377        0.003180
9  691378        0.993821
